# QuickOps — Exploratory Data Analysis
E-Commerce Operations & Business Analytics

**Dataset:** Sample Superstore order-line export (9,994 rows, 21 columns, 2014–2017, United States)

Every section below is written to answer a specific business question, not just "make a chart." Business interpretation follows each major analysis.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import sys
sys.path.insert(0, '../src')
import metrics as m

pd.set_option('display.max_columns', 30)
df = pd.read_csv('../data/processed/orders_clean.csv', parse_dates=['order_date', 'ship_date'])
df.shape


(9994, 24)

## 1. Dataset Overview

In [2]:
print(f"Rows: {len(df)}, Columns: {df.shape[1]}")
print(f"Distinct orders: {df['order_id'].nunique()}")
print(f"Date range: {df['order_date'].min().date()} to {df['order_date'].max().date()}")
df.head()


Rows: 9994, Columns: 24
Distinct orders: 5009
Date range: 2014-01-03 to 2017-12-30


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,delivery_days,data_quality_flag,is_loss_line
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,3,ok,False
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,3,ok,False
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,4,ok,False
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,7,ok,True
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,7,ok,False


## 2. Data Types

In [3]:
df.dtypes

row_id                        int64
order_id                        str
order_date           datetime64[us]
ship_date            datetime64[us]
ship_mode                       str
customer_id                     str
customer_name                   str
segment                         str
country                         str
city                            str
state                           str
postal_code                   int64
region                          str
product_id                      str
category                        str
sub_category                    str
product_name                    str
sales                       float64
quantity                      int64
discount                    float64
profit                      float64
delivery_days                 int64
data_quality_flag               str
is_loss_line                   bool
dtype: object

## 3. Missing Values

In [4]:
missing = df.isnull().sum()
missing[missing > 0] if missing.sum() > 0 else print('No missing values (verified in src/data_cleaning.py).')


No missing values (verified in src/data_cleaning.py).


## 4. Duplicate Analysis

In [5]:
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate row_id values (should be 0, row_id is the natural key):', df['row_id'].duplicated().sum())


Exact duplicate rows: 0
Duplicate row_id values (should be 0, row_id is the natural key): 0


## 5. Data-Quality Validation
Cross-check that the cleaning step's logical validation (ship_date >= order_date) holds.

In [6]:
print(df['data_quality_flag'].value_counts())
assert (df['ship_date'] >= df['order_date']).all(), 'Found ship_date before order_date!'
print('All ship dates occur on/after their order date. ✅')


data_quality_flag
ok    9994
Name: count, dtype: int64
All ship dates occur on/after their order date. ✅


## 6. Distribution Analysis — Sales, Quantity, Profit, Discount

In [7]:
fig = px.histogram(df, x='sales', nbins=60, title='Distribution of Sales (line-item level)', log_y=True)
fig.show()


In [8]:
fig = px.histogram(df, x='profit', nbins=60, title='Distribution of Profit (line-item level)', log_y=True)
fig.show()
print('Negative-profit (loss-making) line items:', (df['profit'] < 0).sum(), f"({100*(df['profit']<0).mean():.1f}% of all line items)")


Negative-profit (loss-making) line items: 1871 (18.7% of all line items)


**Business interpretation:** Sales and Profit are both heavily right-skewed with a long tail of high-value orders — typical of a B2B/B2C mixed retailer. Nearly 1 in 5 line items lose money (18.7%, line-item grain); at the order level — summing profit across each order's line items first — 20.4% of distinct orders are net loss-making. Both numbers are used later, at the grain appropriate to each analysis (see README's KPI Definitions), as a proxy for order-level operational/commercial failure since there is no cancellation field in this dataset.

## 7. Univariate Analysis — Categorical Fields

In [9]:
for col in ['category', 'sub_category', 'region', 'segment', 'ship_mode']:
    print(df[col].value_counts(), '\n')


category
Office Supplies    6026
Furniture          2121
Technology         1847
Name: count, dtype: int64 

sub_category
Binders        1523
Paper          1370
Furnishings     957
Phones          889
Storage         846
Art             796
Accessories     775
Chairs          617
Appliances      466
Labels          364
Tables          319
Envelopes       254
Bookcases       228
Fasteners       217
Supplies        190
Machines        115
Copiers          68
Name: count, dtype: int64 

region
West       3203
East       2848
Central    2323
South      1620
Name: count, dtype: int64 

segment
Consumer       5191
Corporate      3020
Home Office    1783
Name: count, dtype: int64 

ship_mode
Standard Class    5968
Second Class      1945
First Class       1538
Same Day           543
Name: count, dtype: int64 



## 8. Bivariate Analysis — Discount vs Profit

In [10]:
fig = px.scatter(df.sample(2000, random_state=1), x='discount', y='profit', color='category',
                 title='Discount vs Profit (sample of 2,000 line items)', opacity=0.5)
fig.show()


**Business interpretation:** Profit collapses and turns negative as discount increases past roughly 20–30%. This is a directly observable pattern (FACT), not yet a proven causal claim — but it is consistent across categories, which strengthens the case for reviewing discount policy.

## 9. Correlation Analysis

In [11]:
num_cols = ['sales', 'quantity', 'discount', 'profit', 'delivery_days']
corr = df[num_cols].corr()
fig = px.imshow(corr, text_auto='.2f', title='Correlation Matrix (numeric fields)', color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
fig.show()


**Business interpretation:** Discount correlates negatively with profit — the strongest relationship in the matrix. Delivery days show negligible correlation with profit or sales, i.e. slower shipping is not (in this data) associated with a revenue or margin penalty.

## 10. Time Analysis — Orders & Revenue Over Time

In [12]:
trend = (df.assign(month=df['order_date'].dt.to_period('M').astype(str))
           .groupby('month').agg(orders=('order_id','nunique'), revenue=('sales','sum')).reset_index())
fig = px.line(trend, x='month', y=['orders','revenue'], title='Monthly Orders & Revenue Trend')
fig.show()


In [13]:
month_demand = df.assign(m=df['order_date'].dt.month).groupby('m')['order_id'].nunique()
fig = px.bar(month_demand, title='Order Volume by Calendar Month (seasonality across all years)')
fig.show()


**Business interpretation:** Order volume rises steadily through the year and peaks in November/December — a clear seasonal (holiday-driven) demand pattern relevant for staffing and inventory planning.
**Note:** the dataset only has order *dates*, not order *times*, so hour-of-day "peak hours" analysis is not possible here — day-of-week and month are used instead (documented in the Dataset Availability Matrix).

## 11. Location Analysis

In [14]:
loc_perf, median_orders, avg_loss = m.location_performance(df)
loc_perf.head(15)


,state,region,total_orders,total_revenue,avg_delivery_days,loss_rate_pct,quadrant
3,California,West,1021,457687.63,3.87,5.48,High Volume / Good Performance - BENCHMARK
30,New York,East,562,310876.27,4.06,4.27,High Volume / Good Performance - BENCHMARK
41,Texas,Central,487,170188.05,3.93,52.57,High Volume / Poor Performance - PRIORITY
36,Pennsylvania,East,288,116511.91,3.88,54.17,High Volume / Poor Performance - PRIORITY
11,Illinois,Central,276,80166.10,4.12,57.25,High Volume / Poor Performance - PRIORITY
45,Washington,West,256,138641.27,3.97,2.73,High Volume / Good Performance - BENCHMARK
33,Ohio,East,236,78258.14,3.45,51.69,High Volume / Poor Performance - PRIORITY
8,Florida,South,200,89473.71,3.95,33.00,High Volume / Poor Performance - PRIORITY
31,North Carolina,South,136,55603.16,4.00,28.68,High Volume / Poor Performance - PRIORITY
20,Michigan,Central,117,76269.61,4.10,0.00,High Volume / Good Performance - BENCHMARK


In [15]:
fig = px.scatter(loc_perf, x='total_orders', y='loss_rate_pct', size='total_revenue', color='quadrant',
                 hover_name='state', title='State Performance Quadrant: Volume vs. Loss-Making Order Rate')
fig.add_vline(x=median_orders, line_dash='dash')
fig.add_hline(y=avg_loss, line_dash='dash')
fig.show()


**Business interpretation:** States in the top-right quadrant (high volume, high loss rate) — notably Texas, Illinois, and Pennsylvania — combine meaningful order volume with a loss-making rate well above the cross-state average. These are the clearest operational priorities identified in this project (see `sql/10_operational_bottlenecks.sql` for the finer-grained state × sub-category version).

## 12. Category / Product Analysis

In [16]:
cat_perf = m.revenue_by(df, 'sub_category')
fig = px.bar(cat_perf.sort_values('total_revenue'), x='total_revenue', y='sub_category', orientation='h',
             color='profit_margin_pct', color_continuous_scale='RdYlGn', title='Revenue & Margin by Sub-Category')
fig.show()


**Business interpretation:** Tables is a top-5 sub-category by revenue but has a **negative** overall profit margin — the clearest example in this dataset of a category that looks healthy on a revenue-only view but is quietly losing money. It is also associated with heavier average discounting than other sub-categories (see `sql/03_revenue_by_category.sql`), though this is a correlational observation, not a proven cause.

## 13. Operational Performance Analysis — Delivery Time by Ship Mode

In [17]:
delivery_bench = m.delivery_benchmark_by_ship_mode(df)
delivery_bench

,ship_mode,total_orders,avg_delivery_days,benchmark_days,pct_over_benchmark
0,First Class,787,2.19,3.0,0.13
1,Same Day,264,0.05,0.0,4.55
2,Second Class,964,3.23,4.0,20.95
3,Standard Class,2994,5.00,6.0,10.29


In [18]:
fig = px.box(df.drop_duplicates('order_id'), x='ship_mode', y='delivery_days', title='Delivery Days Distribution by Ship Mode')
fig.show()


**Business interpretation:** Delivery time behaves exactly as expected by shipping tier (Same Day ≈ 0 days, Standard Class ≈ 5 days), confirming the ship_mode field is reliable. Since there is no promised-delivery-date field, an internal, DATA-DERIVED 75th-percentile benchmark per ship mode is used instead — this is NOT a contractual SLA. Second Class shows the highest share of orders (21%) exceeding its own historical delivery norm, worth investigating for carrier consistency.

## 14. Key Findings Summary

1. **18.7% of order line items are loss-making** (line-item grain); **20.4% of distinct orders** are net loss-making when profit is summed across each order's own line items first (order grain). Neither is a cancellation rate — this dataset has no order_status field; both are used as documented proxies at the grain appropriate to each analysis.
2. **Deep discounting (41%+) is associated with a 100% loss-making LINE rate** (n=933) — but this is concentrated 66% in a single sub-category (Binders), so it reads as a pricing pattern in specific product lines rather than a universal effect. The exact 0%→100% jump is also consistent with this sample dataset's profit field being formula-driven rather than fully independent transactional noise — correlation is established here, causation is not.
3. **Texas, Illinois, and Pennsylvania are high-volume states with order-level loss rates of 52.6%, 57.2%, and 54.2%** respectively, vs. a 10.5% average across all states — and this is very likely the *same* underlying pattern as Finding #2, not an independent cause: these same three states average 33–39% discount vs. 5–7% in California/New York.
4. **Tables (Furniture) has a negative profit margin** despite being a top-5 revenue sub-category — a "hidden" margin problem invisible on a revenue-only dashboard.
5. **Demand is seasonal**, peaking around November/December — relevant to staffing and inventory planning.
6. **Delivery time varies as expected by ship mode**, and Second Class shows the highest rate of orders exceeding its own internal, data-derived benchmark (not a contractual SLA — no promised-delivery-date field exists in this dataset).

See `README.md` for the full Dataset Availability Matrix, and `INTERVIEW_GUIDE.md` for how each finding translates into a recommendation and a KPI to monitor.
